In [58]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

from statsmodels.tsa.arima.model import ARIMA

In [59]:
df = pd.read_csv("../data/processed/clean_stock_data.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"])

df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change()
df = df.dropna()

In [60]:
returns = df.pivot(index="Date", columns="Ticker", values="Daily_Return")
returns = returns.dropna()

returns.head()

Ticker,BND,SPY,TSLA
Date,,,
2015-01-06,0.002895,-0.009419,0.005664
2015-01-07,0.000602,0.012461,-0.001562
2015-01-08,-0.001563,0.017745,-0.001564
2015-01-09,0.001686,-0.008014,-0.018802
2015-01-12,0.001322,-0.007834,-0.021533


In [61]:
train_size = int(len(returns) * 0.8)

train = returns.iloc[:train_size]
test = returns.iloc[train_size:]

print(train.shape, test.shape)

(2308, 3) (578, 3)


In [62]:
arima_preds = pd.DataFrame(index=test.index)

for col in train.columns:
    model = ARIMA(train[col], order=(5,0,1))
    model_fit = model.fit()

    forecast = model_fit.forecast(steps=len(test))
    arima_preds[col] = forecast

In [63]:
arima_preds = arima_preds.dropna()
test_clean = test.loc[arima_preds.index]

In [64]:
test_reset = test.reset_index(drop=True)
arima_preds = arima_preds.reset_index(drop=True)

In [65]:
print(test_reset.shape)
print(arima_preds.shape)

(578, 3)
(0, 3)


In [68]:
print("TEST SHAPE:", test.shape)
print("ARIMA SHAPE:", arima_preds.shape)
print(arima_preds.head())

TEST SHAPE: (578, 3)
ARIMA SHAPE: (0, 3)
Empty DataFrame
Columns: [BND, SPY, TSLA]
Index: []


In [69]:
common_index = test.index.intersection(arima_preds.index)

test_aligned = test.loc[common_index]
pred_aligned = arima_preds.loc[common_index]

In [71]:
from statsmodels.tsa.arima.model import ARIMA
import pandas as pd

arima_preds = pd.DataFrame(index=test.index)

for col in train.columns:
    
    model = ARIMA(train[col].dropna(), order=(5,0,1))
    model_fit = model.fit()

    forecast = model_fit.forecast(steps=len(test))

    # 🔥 CRITICAL FIX: assign proper index
    forecast = pd.Series(forecast.values, index=test.index)

    arima_preds[col] = forecast

In [72]:
print(arima_preds.head())
print(arima_preds.isna().sum())

                 BND       SPY      TSLA
Date                                    
2024-03-08 -0.000522  0.000073 -0.001105
2024-03-11 -0.000182  0.001628  0.003419
2024-03-12  0.000357 -0.000385  0.002961
2024-03-13  0.000176  0.000112  0.002718
2024-03-14  0.000004  0.000751  0.001411
BND     0
SPY     0
TSLA    0
dtype: int64


In [73]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

for col in test.columns:
    
    y_true = test[col].dropna()
    y_pred = arima_preds[col].dropna()

    min_len = min(len(y_true), len(y_pred))

    y_true = y_true.iloc[:min_len]
    y_pred = y_pred.iloc[:min_len]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"{col}")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("-"*30)

BND
RMSE: 0.0030102262107889882
MAE: 0.0023412353202430184
------------------------------
SPY
RMSE: 0.010241883202923192
MAE: 0.006733971822332401
------------------------------
TSLA
RMSE: 0.03818854240748027
MAE: 0.02785948369584403
------------------------------


In [74]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

metrics = []

for col in test.columns:

    y_true = test[col].dropna()
    y_pred = arima_preds[col].dropna()

    min_len = min(len(y_true), len(y_pred))

    y_true = y_true.iloc[:min_len]
    y_pred = y_pred.iloc[:min_len]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    metrics.append({
        "Model": "ARIMA",
        "Asset": col,
        "RMSE": rmse,
        "MAE": mae
    })

metrics_df = pd.DataFrame(metrics)

metrics_df

,Model,Asset,RMSE,MAE
0,ARIMA,BND,0.003010,0.002341
1,ARIMA,SPY,0.010242,0.006734
2,ARIMA,TSLA,0.038189,0.027859


In [75]:
metrics_df.to_csv("../outputs/forecast_metrics.csv", index=False)